In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
import yfinance as yf

###################################################
#Functions


#Fourier function: given an array of data it gives back the MLE-estimated parameters for the volatility model and for the trend, given a Ornstein-Uhlenbeck stochastic volatility model
def Fourier(P):
    T = 1
    n = len(P)
    dt = T / n
    u = pd.Series(P)
    du = u.diff().dropna().values
    kappa = int(np.round(n * 0.6)) #We use 60% of data quantity as number of harmonics
    N = np.round(kappa / 2).astype(int)
    radiant = np.linspace(0, 2 * np.pi, n - 1, endpoint = False) #Equally-distributed point in the interval [0,2pi)
    #Step 1
    k = np.arange(1, kappa)[:, None]
    matrix = k * radiant
    a = 1 / np.pi * np.cos(matrix) @ du  #With this matrix multiplication we get directly the integrals for each harmonic
    b = 1 / np.pi * np.sin(matrix) @ du
    #Step 2
    def a_star(k):
      if k == 0: return 0
      return(a[abs(k) - 1])
    def b_star(k):
      if k == 0: return 0
      return(abs(k) / k * b[abs(k) - 1])
    def phi(x):
      return(np.sin(x) ** 2 / x ** 2)
    ak = []
    bk = []
    for k in range(0, kappa):
        Sa = 0
        Sb = 0
        for s in range(- N, N - k + 1):
              Sa += a_star(s) * a_star(s + k) + b_star(s) * b_star(s + k) #For any given k, we sum all the values from - N to N - k
              Sb += a_star(s) * b_star(s + k) - b_star(s) * a_star(s + k)
        ak.append(Sa * np.pi / (2 * N + 1)) #The vecotr contains a_k for k = 1, ..., kappa - 1
        bk.append(Sb * np.pi / (2 * N + 1))
    #Step 3
    delta_fourier = 0.4 #Smoothing parameter
    S = np.zeros(n) + (ak[0] / 2)
    span = np.linspace(0, 2 * np.pi, n, endpoint = False)[:, None]
    harmonic_points = span * np.arange(1, kappa)
    harmonic_delta = phi(np.arange(1, kappa) * delta_fourier)
    S += np.cos(harmonic_points) @ (ak[1:] * harmonic_delta) + np.sin(harmonic_points) @ (bk[1:] * harmonic_delta) #The matrix multiplication allows to multiply each cos(k * x_i) times phi(k * delta) * a_k(sigma ** 2) and then to sum them all up, so we have for each entry the complete series evaluated at the point x_i
    p = S * 2 * np.pi / T
    vol_stimata = np.sqrt(p) #Estimation of the volatility series
    Y_stima = np.log(vol_stimata ** 2) #Estimation of the underlying Ornstein-Uhlenbeck process
    Y_early = Y_stima[:-1].sum()
    Y_late = Y_stima[1:].sum()
    Y_mixed = (Y_stima[:-1] * Y_stima[1:]).sum()
    Y_squared = (Y_stima[:-1] ** 2).sum()
    #Various MLE estimates
    speed_stima = 1 / dt * (1 - (Y_late * Y_early - (n - 1) * Y_mixed) / (Y_early ** 2 - (n - 1) * Y_squared))
    mean_stima = - 1 / (speed_stima * dt) * (Y_late * Y_squared - Y_early * Y_mixed) / (Y_early ** 2 - (n - 1) * Y_squared)
    vov_stima = np.sqrt(1 / T * ((Y_stima[1:] - (speed_stima * mean_stima * dt + (1 - speed_stima * dt) * Y_stima[:-1])) ** 2).sum())
    inv_vol_sq = 1.0 / (vol_stimata[:-1] ** 2)
    drift_log = (du / (vol_stimata[:-1] ** 2)).sum() / inv_vol_sq.sum()
    den = Y_early ** 2 - (n - 1) * Y_squared
    num = Y_late * Y_squared - Y_early * Y_mixed
    mu_stima = (1 / dt) * drift_log + 0.5 * (vol_stimata[:-1] ** 2).mean()
    return(speed_stima, mean_stima, vov_stima, mu_stima, Y_stima) #kappa, m, xi, mu, underlying series


#This function takes the estimated parameters of the model, the starting point and other parameters as the number of simulations and the (fixed) random noise to compute the simulations
def ISVaR_Theta(parameters_one, parameters_two, parameters_unnormal_z, horizon, alpha, nsim_candidate, sign, theta, noise, T = 1):
    n = len(parameters_one[4])
    dt = T / n
    kappa_one = parameters_one[0]
    kappa_two = parameters_two[0]
    m_one = parameters_one[1]
    m_two = parameters_two[1]
    xi_one = parameters_one[2]
    xi_two = parameters_two[2]
    mean_one = parameters_one[3]
    mean_two = parameters_two[3]
    last_m = parameters_one[4][-1] #Starting first-underlying-process point
    last_n = parameters_two[4][-1] #Starting second-underlying-process point
    z_horizon = []
    corr = float(parameters_unnormal_z[0])
    last_z = float(parameters_unnormal_z[1]) #Starting spread point
    for i in range(0, nsim_candidate):
      z_path = last_z
      under_process_one = last_m
      under_process_two = last_n
      for t in range(0, horizon):
        #Building of the simulations up to the horizon chosen with the parameters estimated before
        under_process_one += kappa_one * (m_one - under_process_one) * dt + xi_one * np.sqrt(dt) * noise[0, i, t]
        under_process_two += kappa_two * (m_two - under_process_two) * dt + xi_two * np.sqrt(dt) * noise[1, i, t]
        mean = (mean_one - mean_two - 0.5 * np.exp(under_process_one) + 0.5 * np.exp(under_process_two))
        vol = np.sqrt(max(np.exp(under_process_one) + np.exp(under_process_two) - 2 * corr * np.exp(0.5 * (under_process_one + under_process_two)), 1e-5))
        z_path += (mean - theta * vol) * dt + vol * np.sqrt(dt) * noise[2, i, t] #The spread is discretized into little intervals which are summed at each step
      z_horizon.append(z_path) #Vector with the spread at instant "horizon"
    z_horizon = np.array(z_horizon)
    VaR_candidate = np.quantile(z_horizon * sign, 1 - alpha) * sign #Extraction of the quantile, "sign" makes sure to extract the upper quantile when we are interested in the upper bound and the lower one in the other case
    return([VaR_candidate, z_horizon])


#The function is a placeholder to find its zero with respect to VaR, so that we can find the optimal candidate for VaR
def Girsanov_weighted(VaR, sample, theta, noise_z_last, horizon, sign, dt):
    value = (np.where(sample * sign <= VaR * sign, 1, 0) * np.exp(theta * np.sqrt(dt) * noise_z_last - 0.5 * theta ** 2 * dt * horizon)).mean() #Mean using the probability measure given by the Radon-Nikodym derivative from Girsanov's theorem
    return(value)


#This function computes the quantile used to validate the trade
def ISVaR(parameters_one, parameters_two, parameters_unnormal_z, horizon, alpha, nsim_candidate, nsim_IS, sign, T = 1):
    dt = T / len(parameters_one[4])
    noise_1 = np.random.normal(size = (3, nsim_candidate, horizon)) #First random noise used to estimate the quantile with usual Monte Carlo
    VaR_candidate, sample_horizon = ISVaR_Theta(parameters_one, parameters_two, parameters_unnormal_z, horizon, alpha, nsim_candidate, sign, 0, noise_1, T)
    np.random.seed(3456)
    noise_2 = np.random.normal(size = (3, nsim_IS, horizon)) #Second ranom noise used to estimate the value theta that makes the function equal to the candidate, i.e. the optimal shift
    f = lambda x: ISVaR_Theta(parameters_one, parameters_two, parameters_unnormal_z, horizon, alpha, nsim_IS, sign, x, noise_2, T)[0] - VaR_candidate
    theta_star = fsolve(f, x0 = 0)[0] #Find the zero of the function, i.e. optimal theta
    sample_horizon_cal_Gir = ISVaR_Theta(parameters_one, parameters_two, parameters_unnormal_z, horizon, alpha, nsim_IS, sign, theta_star, noise_2, T)[1]
    f = lambda x: Girsanov_weighted(x, sample_horizon_cal_Gir, theta_star, noise_2[2, :, :].sum(axis = 1), horizon, sign, dt) - (1 - alpha) #Alpha indicates the level desired
    VaR = fsolve(f, x0 = VaR_candidate)[0]#Find the zero of the function, hence the optimal VaR
    return(float(VaR))


#A bridge function to pack together the estimations of the two underlying processes and the spread process
def ISVaR_bridge(kappa_one, m_one, xi_one, mean_one, series_one, kappa_two, m_two, xi_two, mean_two, series_two, corr, last_z, horizon, alpha, nsim_candidate, nsim_IS, sign, dt = 1):
    horizon = int(horizon)
    nsim_candidate = int(nsim_candidate)
    nsim_IS = int(nsim_IS)
    parameters_one = [kappa_one, m_one, xi_one, mean_one, series_one]
    parameters_two = [kappa_two, m_two, xi_two, mean_two, series_two]
    parameters_unnormal_z = [corr, last_z]
    return(ISVaR(parameters_one, parameters_two, parameters_unnormal_z, horizon, alpha, nsim_candidate, nsim_IS, sign, dt))


#The true function
def Trading_Strategy(number_one, number_two, train, horizon, stop_loss = 0.05, alpha = 0.2, nsim_candidate = 100, nsim_IS = 500, var_fee = 0, fix_fee = 0, thresh = 2, tolerance = 1.5):
    x = np.log(number_one) - np.log(number_two) #Spread definition
    path = pd.DataFrame({"First" : number_one, "Second" : number_two, "Spread" : np.log(number_one) - np.log(number_two)})
    mean = path["Spread"].rolling(train).mean()
    std = path["Spread"].rolling(train).std()
    path["Standardized"] = (path["Spread"] - mean) / std #Standardization of the series using rolling average and standard error
    path["Signal_1"] = np.where(path["Standardized"] > thresh, 1, np.where(path["Standardized"] < -thresh, -1, 0)) #Possible divergence interval
    indexes = np.where((path["Signal_1"].abs().diff() == 1) | (path["Signal_1"].diff().abs() == 2))[0] #Candidate points of entrance of the strategy (from 0 to 1, from 0 to -1, from 1 to -1 and from -1 to 1)
    signs = np.sign(path.loc[indexes, "Signal_1"]) #Signs to use in the VaR estimation function
    path["Signal_2"] = np.full(len(number_one), np.nan, dtype = float)
    print(f"There are {len(indexes)} possible entries of the strategy")
    cases = len(indexes)
    window = np.arange(-train + 1, 1) #Training widow, the vector is composed of data "train" positions before the entry up to the entry-th
    index_matrix = indexes[:, None] + window #In one matrix we habe all the indexes of the entries we are interested in,each row is a possible entry point
    first_trains = np.log(path["First"].values)[index_matrix] #Log of the intervals of interest
    second_trains = np.log(path["Second"].values)[index_matrix]
    spread_trains = path["Spread"].values[index_matrix]
    a, b, c, d, e = zip(* map(Fourier, first_trains)) #Multivariable function that applies Fourier function on each row, so that each output has for every row the same output of Fourier function on the same row
    temp_one = [a, b, c, d, e]
    a, b, c, d, e = zip(* map(Fourier, second_trains))
    temp_two = [a, b, c, d, e]
    temp_z = [pd.DataFrame(first_trains).diff(axis = 1).corrwith(pd.DataFrame(second_trains).diff(axis = 1), axis = 1).to_numpy(), spread_trains[:, -1]] #Vector with correlation estimation on each row and last point of spread, to start the simulations from
    VaRs = list(map(ISVaR_bridge, temp_one[0], temp_one[1], temp_one[2], temp_one[3], temp_one[4], temp_two[0], temp_two[1], temp_two[2], temp_two[3], temp_two[4], list(temp_z[0]), temp_z[1], list(horizon * np.ones(cases)), list(alpha * np.ones(cases)),
                    list(nsim_candidate * np.ones(cases)), list(nsim_IS * np.ones(cases)), list(signs), list(np.ones(cases)))) #Apply the multivariable function on each row of possible entry point, so that the result is a vector whose entries are the quantiles of interest to validate the entry position
    path.loc[indexes, "Signal_2"] = VaRs
    path["Signal_3"] = np.where(path["Signal_2"] * signs < (path["Spread"] + signs * tolerance * std) * signs, 1, 0) #Validation criterion
    print(f"There have been {path["Signal_3"].sum()} actual trades") #Number of cases validated
    path["Signal_3"] = np.where((path["Signal_3"] == 0) & (path["Signal_1"].diff() != 0), 2, path["Signal_3"]) #Marker where proper trades end
    path["Signal_3"] = path["Signal_3"].replace(0, np.nan).ffill() #Fill ones when the proper trade is executed andtwo otherwise
    path["Signal_3"] = np.where(path["Signal_3"] != 1, 0, 1) #Place back zero when the strategy is not performed
    path["Entry"] = ((path["Signal_1"].abs().diff() == 1) | (path["Signal_1"].diff().abs() == 2)) & (path["Signal_3"]) #Marking each new entry
    path["Trading_Set"] = np.cumsum(path["Entry"]) #Progressive trade label
    path["Entry_Price_x"] = np.where(path["Entry"] == True, path["First"], np.nan) #Place the entry price
    path["Entry_Price_y"] = np.where(path["Entry"] == True, path["Second"], np.nan)
    path["Entry_Price_x"] = path["Entry_Price_x"].ffill() #Roll the entry price
    path["Entry_Price_y"] = path["Entry_Price_y"].ffill()
    path["Stop_Price_x"] = np.where(path["Signal_1"] == 1, path["Entry_Price_x"] * (1 + stop_loss), np.where(path["Signal_1"] == -1, path["Entry_Price_x"] * (1 - stop_loss), 0)) #Stop-loss trigger price for either a long or short position
    path["Stop_Price_y"] = np.where(path["Signal_1"] == 1, path["Entry_Price_y"] * (1 - stop_loss), np.where(path["Signal_1"] == -1, path["Entry_Price_y"] * (1 + stop_loss), 0))
    path["Stop_x"] = ((path["Signal_1"] == 1) & (path["First"] >= path["Stop_Price_x"])) | ((path["Signal_1"] == -1) & (path["First"] <= path["Stop_Price_x"])) #Where the stop-loss order is triggered
    path["Stop_y"] = ((path["Signal_1"] == 1) & (path["Second"] <= path["Stop_Price_y"])) | ((path["Signal_1"] == -1) & (path["Second"] >= path["Stop_Price_y"]))
    path["Stops_x"] = np.where(path["Stop_x"], path["Trading_Set"], 0) #Mark the trading set in which the stop-loss order was triggered
    path["Stops_y"] = np.where(path["Stop_y"], path["Trading_Set"], 0)
    path["Last_Stop_x"] = np.maximum.accumulate(path["Stops_x"]) #Number of trading set when the last stop-loss order occurred
    path["Last_Stop_y"] = np.maximum.accumulate(path["Stops_y"])
    path["Position_x"] = np.where((path["Signal_1"].abs().astype(bool) & (path["Trading_Set"] != path["Last_Stop_x"]) & path["Signal_3"]), np.where(path["Signal_1"] == 1, -1 , np.where(path["Signal_1"] == -1, 1, 0)), 0) #Long or short position if the standardized spread is out of bounds AND the trade has been validated AND a stop-loss order has not been triggered
    path["Position_y"] = np.where((path["Signal_1"].abs().astype(bool) & (path["Trading_Set"] != path["Last_Stop_y"]) & path["Signal_3"]), np.where(path["Signal_1"] == 1, 1 , np.where(path["Signal_1"] == -1, -1, 0)), 0)
    path.loc[len(path["First"]) - 1, "Position_x"] = 0 #We want to close each position at the end of the day
    path.loc[len(path["Second"]) - 1, "Position_y"] = 0
    path["Signal_x"] = path["Position_x"].astype(int).diff().fillna(0) #Buy or sell signal
    path["Signal_y"] = path["Position_y"].astype(int).diff().fillna(0)
    path["Return_x"] = path["Position_x"].shift(1) * path["First"].pct_change() #Return per each time unit in which the strategy is undertaken
    path["Return_y"] = path["Position_y"].shift(1) * path["Second"].pct_change()
    path["Fee_x"] = np.where(path["Signal_x"] != 0, var_fee + (fix_fee / path["First"]), 0) #Compute the fees, both fixed and variable
    path["Fee_y"] = np.where(path["Signal_y"] != 0, var_fee + (fix_fee / path["Second"]), 0)
    path["Return_x"] -= path["Fee_x"] #Consider the fees (subtract)
    path["Return_y"] -= path["Fee_y"]
    path["Return"] = 0.5 * (path["Return_x"] + path["Return_y"]) #It is an equally-weighted strategy, so at each time two positions are opended half the money is placed in each position
    set_return = path.groupby(["Trading_Set"])["Return"].apply(lambda x : (1 + x).prod() - 1) #Compund return per trading set
    return(path, set_return)

###################################################
#Code execution

#You choose the correlated stocks by their ticker
ticker_1 = "UPS"
ticker_2 = "FDX"


df_total = yf.download([ticker_1, ticker_2], period = "4d", interval = "1m", group_by = "ticker") #Data download
df_total = df_total.dropna()   #There could be some missing ticks due to some lack of data

#Remove timezones to avoid computational errors
df_total.index = df_total.index.tz_localize(None) if df_total.index.tz is not None else df_total.index

#Dictionaries for the two stocks
daily_data_1 = {}
daily_data_2 = {}
df_t1 = df_total[ticker_1]
df_t2 = df_total[ticker_2]
for date, intra_data in df_t1.groupby(df_t1.index.date):
    daily_data_1[str(date)] = intra_data.copy() #For each day we add a new entry to the dictionary with thick-by-thick data for that particular day
for date, intra_data in df_t2.groupby(df_t2.index.date):
    daily_data_2[str(date)] = intra_data.copy()


daily_vecs = []
#Select common days in case there are misalignments and order them forward
common_days = sorted(list(set(daily_data_1.keys()) & set(daily_data_2.keys())))[-7:]

for day in common_days:
    vettore_a = daily_data_1[day]["Close"].values #Closing Prices of Stock 1 within 1-minute frame
    vettore_b = daily_data_2[day]["Close"].values #Closing Prices of Stock 2 within 1-minute frame
    res = Trading_Strategy(number_one = vettore_a, number_two = vettore_b, train = 50, horizon = 10, alpha = 0.2, stop_loss = 0.0001, tolerance = 1.2)[1].values
    multindex = pd.MultiIndex.from_product([[day], range(len(res))], names = ["Date", "Progressive"]) #Couple day and trade number (within that day)
    daily_result = pd.Series(res, index = multindex) #label each trade with its day and its number
    daily_vecs.append(daily_result) #Combine the trades
final_vec = pd.concat(daily_vecs)


plt.figure(figsize = (15, 8))
plt.bar(np.arange(len(final_vec)), final_vec.values * 100, label = "Gains per Trade", color = "orange", width = 0.2) #Plot of per-trade returns

#These lines are needed to print the vetical lines corresponding to diffrent trading days
day_change = final_vec.index.get_level_values("Date")
label_index = []
label_name = []
current_date = None
for i, date in enumerate(day_change):
    if date != current_date: #If the trading day is not the current day (one day passed) we set the new day as current day and draw a vertical line
        label_index.append(i)
        label_name.append(date)
        current_date = date
        plt.axvline(x = i, color = "gray", linestyle = "--", alpha = 0.5)


plt.xticks(label_index, label_name, rotation = 45, ha = 'right')
plt.title(f"Returns of {ticker_1} and {ticker_2}", fontsize = 14, fontweight = "bold")
plt.xlabel("Date")
plt.ylabel("Return(%)")
plt.grid(True, alpha = 0.3, axis = "y")
plt.legend()
plt.tight_layout()
plt.show()
